# Day 10 v3 — Gold: All API Marts (Production — Job Parameters)

**Production notebook. Attach to ADF pipeline or Databricks Job.**

**Source:** `silver-volume/api/` (all 17 Silver Delta tables from Day 8 v4)
**Sink:** `gold-volume/api/` (5 Gold Delta marts)

## Parameters

| Parameter | Source | Required | Default | Notes |
|---|---|---|---|---|
| `load_type` | ADF `baseParameters` | Yes | `incremental` | `full` or `incremental` |
| `run_date` | ADF `baseParameters` | No | yesterday UTC (YYYY-MM-DD) | Filter incremental sessions by session_date |

**Incremental behaviour:** rebuilds all Gold marts filtering sessions where `session_date = run_date`, then MERGEs each mart on its natural key. Full load rebuilds entirely via overwrite.

## Gold marts

| Mart | Grain | MERGE key |
|---|---|---|
| `sessions_enriched` | 1 row/session | `session_id` |
| `revenue_by_station` | 1 row/station/day | `station_id` + `session_date` |
| `customer_summary` | 1 row/customer | `customer_id` |
| `charger_utilisation` | 1 row/charger/day | `charger_id` + `session_date` |
| `maintenance_summary` | 1 row/maintenance event | `event_id` |

In [ ]:
# Cell 1 — Imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime, timezone, timedelta

print('Imports OK')

In [ ]:
# Cell 2 — Parameters
# Only load_type is required from ADF.
# run_date defaults to yesterday UTC — current day sessions are still arriving.

_now  = datetime.now(timezone.utc)
_prev = _now - timedelta(days=1)

def _get_param(key, default):
    try:
        val = dbutils.widgets.get(key).strip()
        return val if val else default
    except Exception:
        return default

LOAD_TYPE = _get_param('load_type', 'incremental')
RUN_DATE  = _get_param('run_date',  _prev.strftime('%Y-%m-%d'))

if LOAD_TYPE not in ('full', 'incremental'):
    raise ValueError(f"load_type must be 'full' or 'incremental', got: {LOAD_TYPE!r}")

print(f'load_type : {LOAD_TYPE}')
print(f'run_date  : {RUN_DATE}')
print('Parameters resolved.')

In [ ]:
# Cell 3 — Constants
SILVER_API = '/Volumes/dbw_ev_intelligence_dev/default/silver-volume/api'
GOLD_API   = '/Volumes/dbw_ev_intelligence_dev/default/gold-volume/api'
PIPELINE   = 'pl_gold_api_all_marts_v3'
RUN_TS     = _now.strftime('%Y-%m-%dT%H:%M:%SZ')

print(f'Silver : {SILVER_API}')
print(f'Gold   : {GOLD_API}')
print(f'RUN_TS : {RUN_TS}')

In [ ]:
# Cell 4 — Helper functions
def read_silver(name):
    return spark.read.format('delta').load(f'{SILVER_API}/{name}')

def folder_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

def write_gold(df, mart_name, merge_keys, partition_cols=None):
    """
    Full load  → overwrite.
    Incremental → MERGE on merge_keys if table exists, else overwrite.
    merge_keys: list of column name strings used in the MERGE condition.
    """
    path = f'{GOLD_API}/{mart_name}'

    if LOAD_TYPE == 'full' or not folder_exists(path):
        writer = df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true')
        if partition_cols:
            writer = writer.partitionBy(*partition_cols)
        writer.save(path)
        return 'overwrite'
    else:
        condition = ' AND '.join([f'target.{k} = source.{k}' for k in merge_keys])
        DeltaTable.forPath(spark, path).alias('target') \
            .merge(df.alias('source'), condition) \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()
        return 'merge'

print('Helpers defined')

In [ ]:
# Cell 5 — Read Silver tables
sessions           = read_silver('sessions')
customers          = read_silver('customers')
vehicles           = read_silver('vehicles')
chargers           = read_silver('chargers')
stations           = read_silver('stations')
tariffs            = read_silver('tariffs')
payments           = read_silver('payments')
complaints         = read_silver('complaints')
maintenance_events = read_silver('maintenance_events')
employees          = read_silver('employees')

# Incremental: filter sessions to run_date only
if LOAD_TYPE == 'incremental':
    sessions = sessions.filter(F.to_date('start_time') == F.lit(RUN_DATE))
    print(f'Incremental filter: session_date = {RUN_DATE}  ({sessions.count()} sessions)')
else:
    print(f'Full load: all sessions ({sessions.count()})')

In [ ]:
# Cell 6 — Build dimension tables (shared across marts)
cust_dim = customers.select(
    'customer_id',
    F.concat_ws(' ', 'first_name', 'last_name').alias('customer_name'),
    F.col('email').alias('customer_email'),
    F.col('city').alias('customer_city'),
    F.col('state').alias('customer_state'),
    F.col('country').alias('customer_country'),
)
veh_dim = vehicles.select(
    'vehicle_id',
    F.col('make').alias('vehicle_make'),
    F.col('model').alias('vehicle_model'),
    F.col('year').alias('vehicle_year'),
    F.col('battery_capacity').alias('battery_capacity_kwh'),
    'range_km',
)
chg_dim = chargers.select(
    'charger_id',
    'charger_type',
    F.col('power_kw').alias('charger_power_kw'),
    'connector_type',
    F.col('status').alias('charger_status'),
)
sta_dim = stations.select(
    'station_id',
    F.col('name').alias('station_name'),
    F.col('city').alias('station_city'),
    F.col('state').alias('station_state'),
    F.col('country').alias('station_country'),
    F.col('latitude').alias('station_lat'),
    F.col('longitude').alias('station_lon'),
)
tar_dim = tariffs.select(
    'tariff_id',
    F.col('name').alias('tariff_name'),
    F.col('price_per_kwh').alias('tariff_price_per_kwh'),
    F.col('price_per_min').alias('tariff_price_per_min'),
)
pay_win = Window.partitionBy('session_id').orderBy(F.col('created_at').desc())
pay_dim = (
    payments
    .withColumn('_rn', F.row_number().over(pay_win)).filter(F.col('_rn') == 1).drop('_rn')
    .select('session_id', 'payment_id', 'amount_aud',
            F.col('status').alias('payment_status'), 'payment_method')
)
emp_dim = employees.select(
    F.col('employee_id').alias('technician_id'),
    F.concat_ws(' ', 'first_name', 'last_name').alias('technician_name'),
    F.col('role').alias('technician_role'),
    'department',
)

print('Dimension tables ready')

In [ ]:
# Cell 7 — Build sessions_enriched
sessions_enriched = (
    sessions
    .join(cust_dim, on='customer_id', how='left')
    .join(veh_dim,  on='vehicle_id',  how='left')
    .join(chg_dim,  on='charger_id',  how='left')
    .join(sta_dim,  on='station_id',  how='left')
    .join(tar_dim,  on='tariff_id',   how='left')
    .join(pay_dim,  on='session_id',  how='left')
    .withColumn('session_date',  F.to_date('start_time'))
    .withColumn('session_year',  F.year('start_time'))
    .withColumn('session_month', F.month('start_time'))
    .withColumn('session_hour',  F.hour('start_time'))
    .withColumn('cost_aud', F.coalesce(
        F.col('amount_aud'),
        (F.col('tariff_price_per_kwh') * F.col('energy_kwh')
         + F.col('tariff_price_per_min') * F.col('duration_minutes')).cast('decimal(10,2)')
    ))
    .withColumn('cost_per_kwh', F.when(
        F.col('energy_kwh').isNotNull() & (F.col('energy_kwh') > 0),
        (F.col('cost_aud') / F.col('energy_kwh')).cast('decimal(8,4)')
    ))
    .withColumn('battery_fill_pct', F.when(
        F.col('battery_capacity_kwh').isNotNull() & (F.col('battery_capacity_kwh') > 0),
        (F.col('energy_kwh') / F.col('battery_capacity_kwh') * 100).cast('decimal(6,2)')
    ))
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_load_type',   F.lit(LOAD_TYPE))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
)

print(f'sessions_enriched: {sessions_enriched.count()} rows')

In [ ]:
# Cell 8 — Build aggregated marts from sessions_enriched

# revenue_by_station
revenue_by_station = (
    sessions_enriched
    .groupBy('station_id', 'station_name', 'station_city', 'station_state',
             'station_country', 'session_date')
    .agg(
        F.count('session_id').alias('total_sessions'),
        F.sum('energy_kwh').cast('decimal(14,4)').alias('total_energy_kwh'),
        F.sum('cost_aud').cast('decimal(14,2)').alias('total_revenue_aud'),
        F.avg('duration_minutes').cast('decimal(8,2)').alias('avg_duration_min'),
        F.avg('cost_per_kwh').cast('decimal(8,4)').alias('avg_cost_per_kwh'),
        F.countDistinct('customer_id').alias('unique_customers'),
        F.countDistinct('charger_id').alias('chargers_used'),
    )
    .withColumn('report_year',  F.year('session_date'))
    .withColumn('report_month', F.month('session_date'))
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_load_type',   F.lit(LOAD_TYPE))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
)

# customer_summary — for incremental, rebuild only affected customers
affected_customers = sessions_enriched.select('customer_id').distinct()
cust_sessions_full = (
    read_silver('sessions')
    .join(affected_customers, on='customer_id', how='inner')
    if LOAD_TYPE == 'incremental'
    else sessions
)
complaint_counts = (
    complaints.groupBy('customer_id')
    .agg(F.count('complaint_id').alias('total_complaints'))
)
customer_summary = (
    cust_sessions_full
    .join(cust_dim, on='customer_id', how='left')
    .join(pay_dim,  on='session_id',  how='left')
    .groupBy('customer_id', 'customer_name', 'customer_email',
             'customer_city', 'customer_state', 'customer_country')
    .agg(
        F.count('session_id').alias('total_sessions'),
        F.sum('energy_kwh').cast('decimal(14,4)').alias('total_energy_kwh'),
        F.sum('amount_aud').cast('decimal(14,2)').alias('total_spend_aud'),
        F.avg('duration_minutes').cast('decimal(8,2)').alias('avg_duration_min'),
        F.min('start_time').alias('first_session_ts'),
        F.max('start_time').alias('last_session_ts'),
        F.countDistinct('station_id').alias('stations_visited'),
        F.countDistinct('vehicle_id').alias('vehicles_used'),
    )
    .join(complaint_counts, on='customer_id', how='left')
    .withColumn('total_complaints', F.coalesce('total_complaints', F.lit(0)))
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_load_type',   F.lit(LOAD_TYPE))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
)

# charger_utilisation
charger_utilisation = (
    sessions_enriched
    .groupBy('charger_id', 'charger_type', 'charger_power_kw', 'connector_type',
             'station_id', 'station_name', 'session_date')
    .agg(
        F.count('session_id').alias('total_sessions'),
        F.sum('duration_minutes').cast('decimal(10,2)').alias('total_active_minutes'),
        F.sum('energy_kwh').cast('decimal(12,4)').alias('total_energy_kwh'),
        F.sum('cost_aud').cast('decimal(12,2)').alias('total_revenue_aud'),
        F.avg('energy_kwh').cast('decimal(8,4)').alias('avg_energy_per_session_kwh'),
        F.countDistinct('customer_id').alias('unique_customers'),
    )
    .withColumn('utilisation_pct',
        F.least(
            (F.col('total_active_minutes') / F.lit(1440) * 100).cast('decimal(6,2)'),
            F.lit(100).cast('decimal(6,2)')
        )
    )
    .withColumn('report_year',  F.year('session_date'))
    .withColumn('report_month', F.month('session_date'))
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_load_type',   F.lit(LOAD_TYPE))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
)

print('All aggregated marts built')

In [ ]:
# Cell 9 — Build maintenance_summary (not session-filtered — always full rebuild)
maintenance_summary = (
    maintenance_events
    .join(sta_dim,  on='station_id',    how='left')
    .join(chg_dim,  on='charger_id',    how='left')
    .join(emp_dim,  on='technician_id', how='left')
    .select(
        'event_id', 'charger_id', 'station_id', 'station_name',
        'station_city', 'station_state',
        'charger_type', 'charger_power_kw',
        'event_type', 'description', 'status',
        'scheduled_date', 'completed_date',
        'technician_id', 'technician_name', 'technician_role', 'department',
        F.when(
            F.col('completed_date').isNotNull() & F.col('scheduled_date').isNotNull(),
            F.datediff('completed_date', 'scheduled_date') * 24
        ).cast('decimal(8,2)').alias('resolution_hours'),
        'updated_at',
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_load_type',   F.lit(LOAD_TYPE))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
)

print(f'maintenance_summary: {maintenance_summary.count()} rows')

In [ ]:
# Cell 10 — Write all Gold marts
mart_configs = [
    {
        'name'          : 'sessions_enriched',
        'df'            : sessions_enriched,
        'merge_keys'    : ['session_id'],
        'partition_cols': ['session_year', 'session_month'],
    },
    {
        'name'          : 'revenue_by_station',
        'df'            : revenue_by_station,
        'merge_keys'    : ['station_id', 'session_date'],
        'partition_cols': ['report_year', 'report_month'],
    },
    {
        'name'          : 'customer_summary',
        'df'            : customer_summary,
        'merge_keys'    : ['customer_id'],
        'partition_cols': None,
    },
    {
        'name'          : 'charger_utilisation',
        'df'            : charger_utilisation,
        'merge_keys'    : ['charger_id', 'session_date'],
        'partition_cols': ['report_year', 'report_month'],
    },
    {
        'name'          : 'maintenance_summary',
        'df'            : maintenance_summary,
        'merge_keys'    : ['event_id'],
        'partition_cols': None,
    },
]

results = []
for mc in mart_configs:
    print(f"  Writing {mc['name']} ...", end=' ')
    mode = write_gold(mc['df'], mc['name'], mc['merge_keys'], mc['partition_cols'])
    rows = mc['df'].count()
    results.append({'mart': mc['name'], 'rows': rows, 'mode': mode})
    print(f"{mode} ({rows} rows)")

print('\nAll Gold marts written.')

In [ ]:
# Cell 11 — Run summary
succeeded = [r for r in results if True]   # all succeed or raise

print('=' * 75)
print('GOLD API ALL MARTS v3 — RUN SUMMARY')
print('=' * 75)
print(f'  load_type : {LOAD_TYPE}')
print(f'  run_date  : {RUN_DATE}')
print(f'  run_ts    : {RUN_TS}')
print()
print(f"  {'MART':<25} {'MODE':<10} {'ROWS':>8}")
print(f"  {'-'*25} {'-'*10} {'-'*8}")
for r in results:
    print(f"  {r['mart']:<25} {r['mode']:<10} {r['rows']:>8}")

print()
print('Gold transformation complete.')

In [ ]:
# Cell 12 — Verify Gold output (spot checks)
print('GOLD VERIFICATION')
print('-' * 50)
for mc in mart_configs:
    path = f"{GOLD_API}/{mc['name']}"
    try:
        df = spark.read.format('delta').load(path)
        print(f"  {mc['name']:<25} rows={df.count():>8}  cols={len(df.columns)}")
    except Exception as e:
        print(f"  {mc['name']:<25} ERROR: {e}")

print('\nTop 5 stations by revenue (all-time):')
spark.read.format('delta').load(f'{GOLD_API}/revenue_by_station') \
    .groupBy('station_name') \
    .agg(F.sum('total_revenue_aud').alias('revenue_aud')) \
    .orderBy(F.col('revenue_aud').desc()).show(5, truncate=False)